# Saffron detection — Colab GPU workflow

Run this notebook on a T4 GPU. It is resumable: completed folds are skipped and the final cell zips the run outputs.

Estimated runtime: ~3–5 hours for the full 5-fold run on a T4, based on the smoke-test warmup and the project defaults. This is an estimate only.


In [ ]:
import os, subprocess, sys, json

print('GPU check:')
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError('No GPU detected. Use Colab T4 or a Linux GPU host. This workflow is not valid on CPU for full training.')
print(r.stdout.strip())

import tensorflow as tf
print('TensorFlow', tf.__version__)
print('GPUs', tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('TensorFlow did not detect a GPU.')


In [ ]:
# Pinned package install (matching the repo's version report)
!pip install -q numpy==2.4.2 pandas==2.2.3 scikit-learn==1.6.1 scipy==1.15.2 pillow==12.2.0 tensorflow==2.20.0 keras==3.13.0 plotly==5.0 streamlit==1.35.0
!pip install -q -r requirements.txt


In [ ]:
# Get the repo and data
!git clone https://github.com/ariktadas144/Saffron-Authentication-Detector.git /content/saffron
%cd /content/saffron

# If the data is in Drive, copy it into data/
# from google.colab import drive
# drive.mount('/content/drive')
# !cp '/content/drive/MyDrive/saffron_data/*' data/

print('Repo ready at', os.getcwd())
print('Data dir exists:', os.path.isdir('data'))


In [ ]:
# Prepare dataset or fix path separators if the manifest was created on Windows
import os, glob, shutil, sys

if not os.path.exists('data/manifest.csv'):
    zips = sorted(glob.glob('data/*.zip'))
    if len(zips) != 2:
        raise RuntimeError(f'Expected 2 zip files in data/ but found {len(zips)}: {zips}')
    os.system(f'{sys.executable} scripts/prepare_dataset.py --src {zips[0]} {zips[1]} --data data --folds 5')
else:
    print('Using existing data/manifest.csv')


In [ ]:
# Fold 0 — default settings, run and resume-safe
!python scripts/run_cv.py --manifest data/manifest.csv --out runs/cv_repro --folds 0 --img-size 256 --batch 16 --epochs-warm 3 --epochs-ft 25 --mc-samples 16 --seed 42


In [ ]:
# Label shuffle sanity check, fold 0
!python scripts/label_shuffle_test.py --manifest data/manifest.csv --out runs/audit/label_shuffle --real-out runs/cv_repro --fold 0 --seeds 101 202 303 --img-size 256 --batch 16 --epochs-warm 3 --epochs-ft 25 --mc-samples 16 --seed 42


In [ ]:
# Folds 1–4 (resume-safe; completed folds are skipped if the cache files already exist)
!python scripts/run_cv.py --manifest data/manifest.csv --out runs/cv_repro --folds 1 2 3 4 --img-size 256 --batch 16 --epochs-warm 3 --epochs-ft 25 --mc-samples 16 --seed 42


In [ ]:
# Final aggregation and collection
!python scripts/collect_results.py --cv-dir runs/cv_repro --shuffle-dir runs/audit/label_shuffle --out runs/audit/collected_phase2_results.csv
!python scripts/run_cv.py --out runs/cv_repro --aggregate-only


In [ ]:
# Zip final results for download
import os, shutil, zipfile
archive = 'runs_for_download.zip'
if os.path.exists(archive):
    os.remove(archive)
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk('runs'):
        for name in files:
            full = os.path.join(root, name)
            arcname = os.path.relpath(full, '.')
            zf.write(full, arcname)
print('Created', archive)
